# C5: Operacje geometryczne i relacje przestrzenne
### Programowanie w GIS — Cwiczenia

---

> **Kurs:** Programowanie w GIS (QGIS)  
> **Cwiczenie:** C5  
> **Powiazany wyklad:** W3  
> **Czas:** ~90 minut  

---

### Cele cwiczenia

Po ukonczeniu tego cwiczenia bedziesz potrafil/a:

- bufforowac geometrie w poprawnych jednostkach (metry przez reprojekcje),
- wykonywac operacje na zbiorach geometrii: przeciecie, suma, roznica,
- sprawdzac relacje przestrzenne miedzy obiektami,
- stosowac wzorzec bbox + dokladne sprawdzenie dla wydajnych zapytan,
- laczyc operacje geometryczne w potok analityczny,
- zapisywac wyniki operacji jako nowe warstwy.

---

### Konwencje

| Oznaczenie | Znaczenie |
|---|---|
| **Zadanie** | Polecenie do samodzielnego wykonania |
| **Wskazowka** | Czytaj tylko jesli utkniesz |
| **Pytanie** | Odpowiedz wpisz w komurce ponizej |
| `# [QGIS]` | Kod do uruchomienia w Konsoli Pythona QGIS |

---

### Przygotowanie

1. Uruchom QGIS z wczytana warstwa panstw Natural Earth (`ne_110m_admin_0_countries`).
2. Wczytaj rowniez warstwe `moje_miasta.gpkg` zapisana w cwiczeniu C4
   (lub stworz nowa warstwe punktowa z miastami wedlug C4).
3. Otworz Konsole Pythona i edytor skryptow.


## Spis tresci

1. [Buforowanie w poprawnych jednostkach](#s1)
2. [Operacje na zbiorach geometrii](#s2)
3. [Relacje przestrzenne](#s3)
4. [Wydajne zapytania przestrzenne](#s4)
5. [Centroid i pointOnSurface](#s5)
6. [Zadanie laczace — potok analityczny](#s6)


---
<a id='s1'></a>

## 1. Buforowanie w poprawnych jednostkach

Bufor obliczany bezposrednio na geometrii w EPSG:4326 daje wynik
w stopniach geograficznych — wartosc `buffer(100)` oznacza
~11 000 km, a nie 100 metrow.
Poprawny wzorzec wymaga reprojekcji do ukladu metrycznego.


### Zadanie 1.1 — Porownanie bufora w stopniach vs metrach

Uruchom ponizszy kod i porownaj wyniki:

```python
from qgis.core import (
    QgsGeometry, QgsPointXY,
    QgsCoordinateReferenceSystem, QgsCoordinateTransform, QgsProject
)

# Punkt: centrum Wroclawia w WGS84
g_geo = QgsGeometry.fromPointXY(QgsPointXY(17.038, 51.108))

# Bufor ZLY — w stopniach
bufor_zly = g_geo.buffer(0.5, 32)
print(f'Bufor 0.5 stopnia:')
print(f'  area()   = {bufor_zly.area():.6f}  (stopnie kw.)')

# Bufor DOBRY — reprojekcja do metrow
crs_geo  = QgsCoordinateReferenceSystem('EPSG:4326')
crs_metr = QgsCoordinateReferenceSystem('EPSG:3857')
do_metr  = QgsCoordinateTransform(crs_geo, crs_metr, QgsProject.instance())
do_geo   = QgsCoordinateTransform(crs_metr, crs_geo, QgsProject.instance())

g_metr = QgsGeometry(g_geo)
g_metr.transform(do_metr)

bufor_metr = g_metr.buffer(50_000, 32)  # 50 km
bufor_geo  = QgsGeometry(bufor_metr)
bufor_geo.transform(do_geo)

print(f'Bufor 50 km (poprawny):')
print(f'  area()   = {bufor_geo.area():.6f}  (stopnie kw.)')
print(f'  szerokosc: {bufor_geo.boundingBox().width():.4f} stopni')
print(f'  wysokosc : {bufor_geo.boundingBox().height():.4f} stopni')
```


**Pytanie 1.1** — Oszacuj ile kilometrow odpowiada buforowi 0.5 stopnia
na szerokosci geograficznej Wroclawia (~51°N).
Wskazowka: 1 stopien dlugosci geo na 51°N to okolo 63 km.
Czy bufor w stopniach jest wiec okragly? Wyjasni dlaczego tak lub nie.


In [1]:
# Twoja odpowiedz:


### Zadanie 1.2 — Bufor panstwa w km

Napisz kod ktory dla wybranego panstwa z warstwy Natural Earth
tworzy bufor o promieniu **200 km** i dodaje go jako warstwe na mapie.

Wymagania:

- reprojekcja przez EPSG:3857 zanim wywolasz `.buffer()`,
- wynik z powrotem w EPSG:4326,
- warstwa na mapie z nazwa `'Bufor_200km_[nazwa_kraju]'`.

Wskazowka: pobierz geometrie panstwa:

```python
layer = QgsProject.instance().mapLayersByName('ne_110m_admin_0_countries')[0]
f = next(f for f in layer.getFeatures() if f['NAME'] == 'Poland')
geom = f.geometry()
```


In [2]:
# Twoj kod:

# [QGIS]


### Zadanie 1.3 — Samodzielne: pierscien buforowy

Stworz **pierscien** (obszar miedzy dwoma buforami) wokol stolicy wybranego
panstwa, reprezentujacy stref miedzy 100 a 300 km od centrum.

Etapy:

1. Pobierz geometrie stolicy z warstwy `moje_miasta` (lub uzyj punktu z wspolrzednych).
2. Stworz bufor wewnetrzny 100 km i bufor zewnetrzny 300 km.
3. Pierscien = bufor_zewnetrzny `.difference()` bufor_wewnetrzny.
4. Dodaj pierscien jako warstwe na mape.
5. W konsoli wyswietl pole powierzchni pierscienia w km2.

Wskazowka do przeliczenia pola:
pole w stopniach kwadratowych nie ma sensu fizycznego.
Oblicz pole na geometrii reproj. do EPSG:3857 i podziel przez 1 000 000.


In [3]:
# Twoj kod:

# [QGIS]


**Pytanie 1.3** — Jaka jest roznica miedzy EPSG:3857 a EPSG:2180
jako ukladem docelowym do buforowania?
Ktory daje dokladniejsze wyniki dla danych dotyczacych Polski i dlaczego?


In [4]:
# Twoja odpowiedz:


---
<a id='s2'></a>

## 2. Operacje na zbiorach geometrii


### Zadanie 2.1 — Przeciecie dwoch buforow

Stworz bufory 500 km wokol Wroclawia i wokol Berlina (13.405, 52.520),
oblicz ich przeciecie i dodaj wszystkie trzy geometrie jako warstwy:

```python
from qgis.core import (
    QgsGeometry, QgsPointXY, QgsFeature,
    QgsVectorLayer, QgsProject,
    QgsCoordinateReferenceSystem, QgsCoordinateTransform
)

PROMIEN_KM = 500
PUNKTY = [
    ('Wroclaw', 17.038, 51.108),
    ('Berlin',  13.405, 52.520),
]

crs_geo  = QgsCoordinateReferenceSystem('EPSG:4326')
crs_metr = QgsCoordinateReferenceSystem('EPSG:3857')
do_metr  = QgsCoordinateTransform(crs_geo, crs_metr, QgsProject.instance())
do_geo   = QgsCoordinateTransform(crs_metr, crs_geo, QgsProject.instance())

def bufor_km(lon, lat, km):
    g = QgsGeometry.fromPointXY(QgsPointXY(lon, lat))
    g.transform(do_metr)
    b = g.buffer(km * 1000, 48)
    b_geo = QgsGeometry(b)
    b_geo.transform(do_geo)
    return b_geo

# Twoj kod: stworz bufory, oblicz przeciecie i dodaj warstwy
```

Po stworzeniu przeciecia wyswietl jego pole powierzchni
obliczone przez `QgsDistanceArea`.


In [5]:
# Twoj kod:

# [QGIS]


**Pytanie 2.1** — Czym rozni sie `intersection()` od `union()`
pod wzgledem pola powierzchni wyniku?
Zapisz rownosc: `area(A) + area(B) = area(union) + area(?)`


In [6]:
# Twoja odpowiedz:


### Zadanie 2.2 — Roznica i roznica symetryczna

Korzystajac z buforow z zadania 2.1 oblicz i zwizualizuj:

1. `roznica_WB` = bufor Wroclawia minus przeciecie
   (obszar tylko Wroclawia, bez czesci wspolnej)
2. `roznica_BW` = bufor Berlina minus przeciecie
   (obszar tylko Berlina, bez czesci wspolnej)
3. `sym_roznica` = roznica symetryczna obu buforow
   (wszystko poza czescia wspolna)

Sprawdz numerycznie:

```python
# Ta rownosc powinna byc spelnionia:
print(abs(sym_roznica.area() - (roznica_WB.area() + roznica_BW.area())) < 1e-10)
```


In [7]:
# Twoj kod:

# [QGIS]


### Zadanie 2.3 — Samodzielne: wspolna strefa trzech miast

Znajdz obszar ktory lezy jednoczesnie w zasiegu 800 km
od Wroclawia, Londynu (-0.128, 51.507) i Stambulu (28.979, 41.013).

Etapy:

1. Stworz trzy bufory 800 km.
2. Oblicz przeciecie wszystkich trzech: `A.intersection(B).intersection(C)`.
3. Sprawdz czy wynik nie jest pusty (`isEmpty()`).
4. Jesli nie jest pusty — dodaj go jako warstwe i wyswietl pole pow. w km2.
5. Jesli jest pusty — zmniejsz promien do wartosci przy ktorej przeciecie
   przestaje byc puste (metoda prob i bledow).


In [8]:
# Twoj kod:

# [QGIS]


**Pytanie 2.3** — Przy jakim promieniu bufor przestaje byc pusty?
Ile wynosi pole powierzchni przeciecia w km2?


In [9]:
# Twoja odpowiedz:


---
<a id='s3'></a>

## 3. Relacje przestrzenne


### Zadanie 3.1 — contains i within

Sprawdz relacje `contains` i `within` dla kilku przypadkow.
Uruchom ponizszy kod i wyjasni kazdy wynik:

```python
from qgis.core import QgsGeometry

# Duzy prostokat
A = QgsGeometry.fromWkt('POLYGON((0 0, 10 0, 10 10, 0 10, 0 0))')

# Maly prostokat calkowicie wewnatrz A
B = QgsGeometry.fromWkt('POLYGON((2 2, 5 2, 5 5, 2 5, 2 2))')

# Prostokat wychodzacy poza A
C = QgsGeometry.fromWkt('POLYGON((8 8, 12 8, 12 12, 8 12, 8 8))')

# Punkt na granicy A
D = QgsGeometry.fromWkt('POINT(5 0)')

# Punkt wewnatrz A
E = QgsGeometry.fromWkt('POINT(5 5)')

testy = [
    ('A.contains(B)',  A.contains(B)),
    ('B.within(A)',    B.within(A)),
    ('A.contains(C)',  A.contains(C)),
    ('A.intersects(C)', A.intersects(C)),
    ('A.contains(D)',  A.contains(D)),
    ('A.contains(E)',  A.contains(E)),
    ('D.within(A)',    D.within(A)),
    ('A.touches(C)',   A.touches(C)),
]

for opis, wynik in testy:
    print(f'  {opis:<25} -> {wynik}')
```


**Pytanie 3.1** — `A.contains(D)` gdzie D lezy na granicy A.
Jaki daje wynik i dlaczego jest to wazne praktycznie?
Jak zachowuje sie `D.within(A)` w tym samym przypadku?


In [10]:
# Twoja odpowiedz:


### Zadanie 3.2 — Znajdz panstwo zawierajace punkt

Napisz funkcje ktora dla podanej pary wspolrzednych
zwraca nazwe panstwa, na terytorium ktorego lezy ten punkt.

```python
def znajdz_panstwo(lon, lat):
    layer = QgsProject.instance().mapLayersByName('ne_110m_admin_0_countries')[0]
    punkt = QgsGeometry.fromPointXY(QgsPointXY(lon, lat))
    for f in layer.getFeatures():
        if f.geometry().contains(punkt):
            return f['NAME']
    return 'Poza ladem'

# Przetestuj dla kilku punktow
print(znajdz_panstwo(17.038, 51.108))   # Wroclaw
print(znajdz_panstwo(0.0,    51.5))      # Londyn
print(znajdz_panstwo(0.0,    0.0))       # Atlantyk
```

Nastepnie rozszerz funkcje tak, zeby uzywala
`QgsFeatureRequest().setFilterRect(bbox)` jako wstepnego filtru
przed sprawdzeniem `contains`.


In [11]:
# Twoja funkcja (wersja podstawowa + zoptymalizowana):

# [QGIS]


**Pytanie 3.2** — Przetestuj obie wersje funkcji (z `setFilterRect` i bez)
na punkcie lezacym na srodku oceanu. Co zwraca kazda wersja i dlaczego?
Ktora jest szybsza dla warstwy z 200 obiektami?


In [12]:
# Twoja odpowiedz:


### Zadanie 3.3 — Samodzielne: ile panstw przecina rownik

Stworz geometrie rownika jako linii:

```python
rownik = QgsGeometry.fromWkt('LINESTRING(-180 0, 180 0)')
```

Nastepnie znajdz wszystkie panstwa ktore sa przecinane przez rownik.
Wyswietl ich nazwy i kontynenty, posortowane alfabetycznie.

Bonusowe pytanie: ile panstw ma wieksze terytorium na polkuli
poludniowej niz polnocnej? (wymaga bardziej zaawansowanej analizy)


In [13]:
# Twoj kod:

# [QGIS]


**Pytanie 3.3** — Ile panstw przecina rownik? Czy wynik Cie zaskoczyly?
Ktore kontynenty sa reprezentowane?


In [14]:
# Twoja odpowiedz:


---
<a id='s4'></a>

## 4. Wydajne zapytania przestrzenne

Sprawdzanie relacji dla kazdej pary obiektow duzej warstwy
ma zlozonosc O(n). Dla wielu zapytan na tej samej warstwie
warto wstepnie odfiltrowac kandydatow przez bounding box —
QGIS uzywa do tego indeksu przestrzennego R-tree.


### Zadanie 4.1 — Wzorzec bbox + dokladne sprawdzenie

Znajdz wszystkie panstwa ktore przecinaja sie z buforem
500 km wokol Wroclawia, uzywajac dwuetapowego wzorca:

```python
from qgis.core import QgsFeatureRequest

# Etap 1: bbox filter (szybki, moze dawac false positives)
req = QgsFeatureRequest().setFilterRect(bufor_geo.boundingBox())
kandydaci = list(layer.getFeatures(req))
print(f'Kandydaci po bbox: {len(kandydaci)}')

# Etap 2: dokladne sprawdzenie geometryczne
wynik = [f for f in kandydaci if f.geometry().intersects(bufor_geo)]
print(f'Po dokladnym sprawdzeniu: {len(wynik)}')
```

Porownaj liczbe kandydatow po pierwszym etapie z liczba
po etapie drugim. Ile obiektow zostalo odfiltrowanych w etapie 2?
Sa to tzw. **false positives** indeksu przestrzennego.


In [15]:
# Twoj kod (z bufor_geo z poprzednich zadan lub nowym):

# [QGIS]


**Pytanie 4.1** — Dlaczego indeks przestrzenny moze zwracac false positives?
Narysuj lub opisz slowami geometrie ktora przeszlaby przez filtr bbox
ale nie przeszlaby przez dokladne sprawdzenie `intersects`.


In [16]:
# Twoja odpowiedz:


### Zadanie 4.2 — Samodzielne: zapytanie z limitem

Napisz funkcje `panstwa_blisko(lon, lat, km, max_wynikow=10)`
ktora zwraca posortowana liste co najwyzej `max_wynikow` panstw
najblizszych podanemu punktowi, wraz z odlegloscia.

Wymagania:

- uzyj wzorca bbox + `intersects` do wstepnego filtrowania,
- odleglosc mierz przez `QgsDistanceArea` (fizyczna, w km),
- wynik: lista krotek `(nazwa_panstwa, odleglosc_km)`,
- posortowana rosnaco wedlug odleglosci,
- obcieta do `max_wynikow` elementow.

Przetestuj dla Wroclawia z `km=2000` i `max_wynikow=5`.


In [17]:
# Twoja funkcja:

# [QGIS]


---
<a id='s5'></a>

## 5. Centroid i pointOnSurface


### Zadanie 5.1 — Centroid poza obiektem

Stworz geometrie dla ktorej centroid pada poza jej obrysem
i zweryfikuj to przez `contains()`:

```python
# Litera 'C' jako uproszczony polygon wklesly
litera_c = QgsGeometry.fromWkt(
    'POLYGON((0 0, 6 0, 6 2, 2 2, 2 8, 6 8, 6 10, 0 10, 0 0))'
)

centroid = litera_c.centroid()
point_on = litera_c.pointOnSurface()

print(f'Centroid        : {centroid.asWkt()}')
print(f'pointOnSurface  : {point_on.asWkt()}')
print(f'Zawiera centroid: {litera_c.contains(centroid)}')
print(f'Zawiera pointOn : {litera_c.contains(point_on)}')
```

Nastepnie stwórz warstwe z trzema obiektami:
geometria litery C, jej centroid (czerwony) i pointOnSurface (zielony).


In [18]:
# Twoj kod:

# [QGIS]


**Pytanie 5.1** — Dla jakiego rodzaju ksztaltow
centroid z pewnoscia bedzie wewnatrz obiektu?
Podaj przynajmniej dwa typy geometrii i wyjasni dlaczego.


In [19]:
# Twoja odpowiedz:


### Zadanie 5.2 — Samodzielne: panstwa z centroidem na morzu

Korzystajac z warstwy panstw Natural Earth,
znajdz wszystkie panstwa europejskie, dla ktorych centroid
NIE lezy wewnatrz ich terytorium.

Etapy:

1. Przefiltruj warstwe do panstw europejskich (`CONTINENT = 'Europe'`).
2. Dla kazdego panstwa sprawdz `geom.contains(geom.centroid())`.
3. Wypisz nazwy panstw z centroidem poza krajem.
4. Dla kazdego takiego panstwa wyswietl:
   - wspolrzedne blednego centroidu,
   - wspolrzedne poprawionego `pointOnSurface()`.
5. Stworz dwie warstwy punktowe: bledne centroidy i poprawione punkty.


In [20]:
# Twoj kod:

# [QGIS]


---
<a id='s6'></a>

## 6. Zadanie laczace — potok analityczny

Ponizsze zadanie wymaga polaczenia wszystkich umiejetnosci z C4 i C5.
Napisz je jako jeden kompletny skrypt w edytorze skryptow QGIS.


### Zadanie 6.1 — Analiza dostepnosci stolic

Masz warstwe panstw Natural Earth i warstwe `moje_miasta` z C4.

Napisz skrypt ktory dla **kazdego miasta** z warstwy `moje_miasta`:

**Krok 1 — Bufor dostepnosci:**  
Stworz bufor 500 km wokol miasta (poprawnie w metrach).

**Krok 2 — Panstwa w zasiegu:**  
Znajdz wszystkie panstwa przecinajace sie z buforem
(wzorzec bbox + `intersects`).

**Krok 3 — Raport:**  
Wyswietl w konsoli:

```
Wroclaw (641 000 mieszkancow):
  Panstw w zasiegu 500 km: 12
  Lacznie ludnosc: 234 567 890
  Lista: Polska, Niemcy, Czechy, ...
```

**Krok 4 — Warstwy wynikowe:**  
Dla kazdego miasta stworz oddzielna warstwe z buforowq
i dodaj ja do projektu.

**Krok 5 — Podsumowanie:**  
Po przejsciu przez wszystkie miasta wyswietl:
- ktore miasto ma najwiecej panstw w zasiegu,
- ktore miasto ma dostep do najwiekszej lacznie populacji.

**Krok 6 — Zapis:**  
Zapisz warstwe buforow wszystkich miast jako jeden plik
`zasieg_miast.gpkg` (kolejne bufory jako kolejne obiekty tej samej warstwy).

Wymagania:

- caly potok jako jeden skrypt z funkcjami,
- obsluga przypadku gdy miasto nie ma zadnych panstw w zasiegu,
- wyniki czytelnie sformatowane w konsoli.


In [21]:
# Twoj kompletny skrypt:

# [QGIS]


**Pytanie 6.1** — Ktore miasto z Twojej warstwy ma dostep
do najwiekszej lacznie populacji w promieniu 500 km?
Czy wynik Cie zaskoczyl? Jak moglbys/moglabys zmodyfikowac
skrypt zeby znalezc 'optimalne' polozenie nowego miasta
maksymalizujace dostep do populacji?


In [22]:
# Twoja odpowiedz:


---

## Podsumowanie C5

W tym cwiczeniu pracowales/pracowałas z:

- buforowaniem w poprawnych jednostkach przez reprojekcje do EPSG:3857,
- operacjami na zbiorach: `intersection`, `union`, `difference`, `symDifference`,
- relacjami przestrzennymi: `contains`, `within`, `intersects`, `touches`,
- wzorcem wydajnego zapytania: bbox filter + dokladne sprawdzenie,
- centroidem i `pointOnSurface` — roznice i przypadki uzycia,
- laczeniem operacji w potok analityczny.

---

### Na C6

Nastepne cwiczenie (C6) jest poswiecone GeoPandas —
analizie danych wektorowych poza QGIS w srodowisku Jupyter.

---
*Cwiczenie C5 — Programowanie w GIS (QGIS) — studia licencjackie.*
